In [ ]:
# ============================================================
# INSTALL
# ============================================================

!pip install -q ultralytics easyocr openpyxl pillow

# ============================================================
# IMPORTS
# ============================================================

import cv2
import pandas as pd
import easyocr
import re
import os

from ultralytics import YOLO

from openpyxl import Workbook
from openpyxl.drawing.image import Image as XLImage
from openpyxl.utils import get_column_letter

# ============================================================
# PATHS
# ============================================================

VIDEO_PATH = "/content/test_video1.mp4"

MODEL_PATH = "/content/best.pt"

OUTPUT_EXCEL = "/content/detection_results.xlsx"

TEMP_IMAGE_DIR = "/content/detection_images"

os.makedirs(TEMP_IMAGE_DIR, exist_ok=True)


model = YOLO(MODEL_PATH)


reader = easyocr.Reader(['en'], gpu=True)



cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)


wb = Workbook()
ws = wb.active

headers = [
    "Frame",
    "Timestamp",
    "Class",
    "Confidence",
    "Latitude",
    "Longitude",
    "Detection Image"
]

ws.append(headers)



frame_idx = 0
excel_row = 2

while True:

    ret, frame = cap.read()

    if not ret:
        break

    overlay = frame[0:120, :]

    ocr_results = reader.readtext(overlay, detail=0)

    full_text = " ".join(ocr_results)



    latitude = None
    longitude = None

    lat_match = re.search(r'Latitude[: ]+([0-9.]+)', full_text)
    lon_match = re.search(r'Longitude[: ]+([0-9.]+)', full_text)

    if lat_match:
        latitude = lat_match.group(1)

    if lon_match:
        longitude = lon_match.group(1)


    results = model.predict(
        frame,
        conf=0.25,
        verbose=False
    )

    result = results[0]

    detections = result.boxes

    timestamp = frame_idx / fps



    for i, box in enumerate(detections):

        cls_id = int(box.cls[0])

        class_name = model.names[cls_id]

        confidence = float(box.conf[0])

        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

        annotated = frame.copy()

        cv2.rectangle(
            annotated,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        label = f"{class_name} {confidence:.2f}"

        cv2.putText(
            annotated,
            label,
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )


        image_path = os.path.join(
            TEMP_IMAGE_DIR,
            f"frame_{frame_idx}_det_{i}.jpg"
        )

        cv2.imwrite(image_path, annotated)

        # ====================================================
        # WRITE EXCEL ROW
        # ====================================================

        ws.cell(excel_row, 1, frame_idx)
        ws.cell(excel_row, 2, round(timestamp, 2))
        ws.cell(excel_row, 3, class_name)
        ws.cell(excel_row, 4, round(confidence, 4))
        ws.cell(excel_row, 5, latitude)
        ws.cell(excel_row, 6, longitude)

        # ====================================================
        # INSERT IMAGE
        # ====================================================

        img = XLImage(image_path)

        img.width = 220
        img.height = 140

        cell_position = f"G{excel_row}"

        ws.add_image(img, cell_position)

        # Increase row height
        ws.row_dimensions[excel_row].height = 110

        excel_row += 1

    frame_idx += 1

# ============================================================
# COLUMN WIDTHS
# ============================================================

column_widths = {
    1: 12,
    2: 15,
    3: 25,
    4: 15,
    5: 20,
    6: 20,
    7: 40
}

for col, width in column_widths.items():

    ws.column_dimensions[
        get_column_letter(col)
    ].width = width

# ============================================================
# SAVE EXCEL
# =================== =========================================

wb.save(OUTPUT_EXCEL)

cap.release()

print("DONE")
print("Saved Excel:")
print(OUTPUT_EXCEL)

model code